### Writing to sentinel-5P zarr store

Necessary imports

In [1]:
import zarr 
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from scipy.interpolate import griddata
from pyproj import Transformer, CRS
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling

import reprojection
import os

For now we will only write CO data to the zarr store. For this we will go through the folder containing our netCDF files and reproject them to EUQI7 grid, as some files don't extent over Austria a potential error is overriden.

In [2]:
qa_value = 0.5
is_no2 = False

In [3]:
path_base = "/home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/"

The datasets in the datasets list are now concatenated, if the timecoordinate is the same for two datasets the data is averaged.

Now we can open the zarr store

The time indexes are then calculated

In [4]:
range(1,1,12)

range(1, 1, 12)

Next the data is written to the correct position in the zarr store. The correct nodata value and scale factor is applied before writing

In [6]:
for i in ["07", "08", "09", "10", "11"]:
    datasets = []
    count = 0
    for file in os.listdir(path_base):
        path = os.path.join(path_base, file)
        if file.startswith(f"S5P_OFFL_L2__CO_____2024{i}"):
            print("File Number:", count)
            count += 1
            try:
                datasets.append(reprojection.reproject_CO(path, qa_value, is_no2))
            except (ValueError) as e:
                print(f"Warning file {file}: {e}")
                continue
        else:
            continue

    merged = reprojection.merge_mean_by_time(datasets)

    product_type = "CO"
    store_path = "/home/simon/eodc_datasync/private/eodc_logs/s5p/s5p.zarr"


    store = zarr.storage.LocalStore(store_path)
    group = zarr.group(store=store, path=product_type)

    time_origin = np.datetime64("2018-04-01")
    time_min = (merged.time.min().values.astype("datetime64[D]") - time_origin).astype("int64")
    time_max = (merged.time.max().values.astype("datetime64[D]") - time_origin).astype("int64")

    for var in merged.data_vars:
        scale = group[var].attrs["scale_factor"]
        fill_value = group[var].attrs["_FillValue"]
        
        # Get the data slice from merged (xarray) with .values
        data = merged[var].isel(time=slice(time_min, time_max)).values
        name = var
        print(merged.var)
        zarr_time = xr.open_zarr(store_path, group="CO", consolidated=True).time
        merged_aligned = merged.reindex(
            time=zarr_time[time_min:time_max],
            method=None  # do NOT interpolate
        )
                
        # Scale and fill NaNs accordingly
        data_scaled = np.nan_to_num(np.round(merged_aligned[var].values / scale), nan=fill_value).astype(group[var].dtype)
        
        # Write into zarr slice
        group[var][time_min:time_max, :, :] = data_scaled


File Number: 0


getfattr: /home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__CO_____20240725T115833_20240725T134003_35144_03_020600_20240728T085221.nc: Operation not supported


File Number: 1


getfattr: /home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__CO_____20240705T012503_20240705T030633_34854_03_020600_20240706T183803.nc: Operation not supported


File Number: 2


...

File Number: 54


getfattr: /home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__CO_____20241112T104321_20241112T122450_36704_03_020701_20241115T050043.nc: Operation not supported


File Number: 55


getfattr: /home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__CO_____20241127T092018_20241127T110148_36916_03_020800_20241128T231142.nc: Operation not supported


<bound method DatasetAggregations.var of <xarray.Dataset> Size: 5MB
Dimensions:                                (time: 30, y: 60, x: 90)
Coordinates:
  * time                                   (time) datetime64[s] 240B 2024-11-...
  * y                                      (y) float64 480B 1.8e+06 ... 1.21e+06
  * x                                      (x) float64 720B 4.5e+06 ... 5.39e+06
Data variables:
    qa_value                               (time, y, x) float64 1MB 1.0 ... 0.0
    carbonmonoxide_total_column            (time, y, x) float64 1MB 0.02816 ....
    carbonmonoxide_total_column_precision  (time, y, x) float64 1MB 0.000732 ...
    carbonmonoxide_total_column_corrected  (time, y, x) float64 1MB 0.02919 ....>
<bound method DatasetAggregations.var of <xarray.Dataset> Size: 5MB
Dimensions:                                (time: 30, y: 60, x: 90)
Coordinates:
  * time                                   (time) datetime64[s] 240B 2024-11-...
  * y                                 